In [ ]:
import pandas as pd
import numpy as np
import time

data = pd.read_parquet("../Dataset/Clean/Dataset_with_clusters.parquet")
sample = data.sample(10)
data.shape

## Entity & Keyword Extraction
This is an important component for textual analysis.
This Component has two key roles:
1. **High-Fidelity Information Extraction** [Involvement of NER]
2. **Semantic Tagging** [About NER and Semantic]

Standard Spacy models are often too general for news, <br>
which frequently contains niche organizations, emerging technologies, or specific political designations.

#### NER Model comparision for News Analysis
| Model        | Technique                         | Key Strength                                                                 | Performance (Speed)        |
|--------------|-----------------------------------|------------------------------------------------------------------------------|----------------------------|
| GLiNER       | Bi-Encoder / Zero-Shot            | Can extract any label you define on the fly without retraining.             | Medium (Better on GPU)     |
| Flair        | Character-Level Embeddings        | Industry standard for accuracy; handles typos and "news-speak" well.        | Slow (Sequential processing) |
| spaCy (TRF)  | Transformer-based (RoBERTa)       | Much stronger than the standard "sm" model; integrates with spaCy ecosystem | Medium-Fast                |
| SpanMarker   | PLM-based (BERT/RoBERTa)          | Current SOTA (State of the Art) for fixed-label NER accuracy.               | Medium                     |


In [ ]:
query = data.query('word_count > 100').sample()['Content'].values[0]
print(query)

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp(query)
for ent in doc.ents:
    print(ent.text, ent.label_)
    
print("Length: ", len(doc.ents))

In [ ]:
nlp = spacy.load("en_core_web_trf")
doc = nlp(query)

print("List of Entities: \n")
for i,ent in enumerate(doc.ents):
    print(f"{i+1})",ent.text, "| Label: ",ent.label_, "-> ",spacy.explain(ent.label_))
    
print("Length: ", len(doc.ents))

spacy.displacy.render(doc, style="ent")

Transformer model of Spacy is little slow to load but it is a good model for NER.

Now, we will try other pre-trained Model which are different from spacy.

#### 1. `SpanMarker` <br>
SpanMarker is a framework for training powerful Named Entity Recognition models using familiar encoders such as BERT, RoBERTa and DeBERTa.

In [ ]:
# nlp = spacy.load("en_core_web_sm", exclude=["ner"])
# nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-bert-base-fewnerd-fine-super"})

# doc = nlp(query)
# print([(entity, entity.label_) for entity in doc.ents])

> Currently, Spanmarker is not compatable wiht current version of transformer. 

#### 2. **`Bert-base NER`**
distilbert-NER is the fine-tuned version of DistilBERT, which is a distilled variant of the BERT model. <br>
DistilBERT has fewer parameters than BERT, making it smaller, faster, and more efficient. distilbert-NER is specifically fine-tuned for the task of Named Entity Recognition (NER).

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained("dslim/distilbert-NER")
model = AutoModelForTokenClassification.from_pretrained("dslim/distilbert-NER")

ner = pipeline("ner", model=model, tokenizer=tokenizer)
res = pd.DataFrame(ner(query))
res[res["entity"].str.contains("B")]

This model is not performing well for our task. This does token level NER where it is missing many entities.

#### 3. **`Flair`**
Flair NER model to recognize 4 or 18 predefined entities.
| tag           |	meaning                 |
| ------------- | ------------------------- |
| CARDINAL      |	cardinal value          |
| DATE          |	date value              |
| EVENT         |	event name              |
| FAC           |	building name           |
| GPE           |	geo-political entity    |
| LANGUAGE      |	language name           |
| LAW           |	law name                |
| LOC           |	location name           |
| MONEY         |	money name              |
| NORP          |	affiliation             |
| ORDINAL       |	ordinal value           |
| ORG           |	organization name       |
| PERCENT       |	percent value           |
| PERSON        |	person name             |
| PRODUCT       |	product name            |
| QUANTITY      |	quantity value          |
| TIME          |	time value              |
| WORK_OF_ART   |	name of work of art     |

In [ ]:
from flair.models import SequenceTagger
from flair.data import Sentence

tagger = SequenceTagger.load("flair/ner-english-ontonotes-fast")

In [ ]:
sent = Sentence(query)

tagger.predict(sent)
# print(sent)

for entity in sent.get_spans("ner"):
    print(entity)

Flair NER is very good but it has high latency which is a problem for real-time application. <br>
High Efficiency but low throughput -> Can be used for smaller article where we need predefined class entities.

#### 4. **`GLiner`**
GLiNER(General Linguistic Named Entity Recognition) is a framework for training and deploying small Named Entity Recognition (NER) models with zero-shot capabilities. <br>
In addition to tradition NER, it also supports joint entity and relation extraction.

GLiner does not have a fixed "output head" for specific labels, it uses a **Bi-Encoder/Span-matching** archtecture.

In [ ]:
from gliner import GLiNER
from gliner2 import GLiNER2

ner_labels = [
    "Person", "Celebrity", "Political party", "Politician", "Activist", "Criminal", "Victim", "Witness",
    "Profession", "Job title", "Author", "Scientist", "Journalist", "Speaker", "Writer", "Artist",
    "Affiliation", "Organisation", "Company", "Startup", "Institution", "College", "University", "Government agency", "Military organisation", "Union", "Sports team", "Media",
    "Country", "City", "State", "Region", "Continent", "Climate zone", "Forest", "Desert", "Mountain", "Park", "Water body",
    "Building", "Airport", "Monument", "Landmark", "Date", "Time", "Duration", "Percent", "Money", "Temperature", "Speed", "Age",
    "Law", "Case", "Judge", "Constitution", "Election", 
    "Medical", "Disease", "Drug", "Symptom", "Chemical"
    "Location", "Product", "Event", "Work_of_art", "Language",
    "Business", "Market", "Stock", "Currency", "Product"
    "Sport", "Games", "Award", "Event",
    "Art", "Book", "Movie", "TV show", 
    "Computer", "Vehicle", "Machine", "Programming language", "Technology",
    "Color", "Shape", "Size", "Weight", "Weapon", "Battle", "Natural disaster",
    "Quantity", "Ordinal", "Cardinal", 
    "Animal", "Plant", "Organism",
]

In [ ]:
# Loading model
gliner_model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

i = 0
start = time.time()
entities = gliner_model.predict_entities(query, labels=ner_labels, threshold=0.5)
# Display predicted entities and their labels
for entity in entities:
    print(entity["text"], "=>", entity["label"])
    i += 1

print(f"Total entities: {i}")
print("Time taken: ", time.time() - start)

In [ ]:
# Loading model
gliner_model = GLiNER2.from_pretrained("fastino/gliner2-large-v1")

start = time.time()
entities = gliner_model.extract_entities(query, entity_types=ner_labels)

i = 0
for label, ent in entities["entities"].items():
    if ent:
        print(f"{label}: {ent}")
        i += len(ent)
        
print(f"Total entities: {i}")
print("Time taken: ", time.time() - start)

Gliner is the best model for Recoginition of Named Entities with custom class label. <br>
It understand the semantic meaning of searched article and each label and finds matching entities.

> Gliner-medium-v2 is faster and better then Gliner2-large-v1

Now, we will extract keywords from our given article. <br>

For keyword extraction, there is one clear cut winner -> 
### ***`KeyBERT`***
KeyBERT is fast library for keyword and keyphrase extraction that utilizes BERT embeddings to find sub-phrases most similar to a document. <br>
It calculates cosine similarity between document and n-gram embeddings to identify the most relevant terms.

##### <b> Key Features </b>
- Methodology: Uses BERT to generate document embeddings, extract N-gram token & compute cosine similarity to rank them.
- Customization: Allows tuning of top-n keywords, N-gram range & diversity algrorithm. [Maximal Marginal Relevance]
- Flexibility: Support various backends(PyTorch, Tensorflow, JAX, Huggign Face) & allows for multilingual models
- Integration: Easily intergrate with other Bert based specialised models
- Performance: Fast & memory efficient

In [ ]:
from keybert import KeyBERT
keybert = KeyBERT(model = 'all-MiniLM-L6-v2')
keywords = keybert.extract_keywords(query, keyphrase_ngram_range=(1, 2), 
    stop_words="english",
    use_mmr=True, # Maximal Marginal Relevance - for diversity
    diversity=0.5, # 0.0 to 1.0
    top_n=10, # Number of keywords
)
keywords

This is the best model for NER+keyword extraction. <br>
We will optimize this model for our project inference pipeline.

Now, There are a lot of various pipelines and libraries out there for NER+keyword extraction but we dont know which one if correct for our task.

Also, NER model is not something we can fine-tune as it is blind implementation.
So, we need to foucs on evaluation for NER.

But, There is no direct correct way for evaluation for NER task. <br>
So, a combination of various method/trick/metrics will be used in combination.

Evaluation Pipeline:
Design a class for NER Evaluation.
1. Intrinsic proxy metrics: Precision, Recall, F1 vs summary-entities
2. Per-class stats: entity distribution vs topic clusters
3. Efficiency: Latency, Throughput
4. Extrinsic metrics: topic coherence delta, clustering purity, keyword precision@k
5. Final combined score

In [ ]:
import time
import numpy as np
from collections import defaultdict, Counter
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy

In [ ]:
class NEREvaluator:
    def __init__(self, model, model_type="spacy"):
        """
        model_type: "spacy" | "HuggingFace" | "flair" | "gliner"
        model: loaded NER model
        """
        self.model = model
        self.model_type = model_type.lower()
        
    # 1. Internal NER call
    def _extract_entities(self, text):

        if self.model_type == "spacy":
            doc = self.model(text)
            return [
                {"start": ent.start_char, "end": ent.end_char, "label": ent.label_, "text": ent.text}
                for ent in doc.ents
            ]

        elif self.model_type == "huggingface":
            outputs = self.model(text)
            return [
                {"start": ent["start"], "end": ent["end"], "label": ent["entity"], "text": ent["word"]}
                for ent in outputs
            ]

        elif self.model_type == "flair":
            from flair.data import Sentence
            sent = Sentence(text)
            self.model.predict(sent)
            return [
                {"start": ent.start_position, "end": ent.end_position, "label": ent.tag, "text": ent.text}
                for ent in sent.get_spans("ner")
            ]

        elif self.model_type == "gliner":
            ents = self.model.predict_entities(text)
            return [
                {"start": ent["start"], "end": ent["end"], "label": ent["label"], "text": ent["text"]}
                for ent in ents
            ]

        else:
            raise ValueError("Unsupported model_type")
        
    # 2. Run model
    def run_model(self, df, text_col="Content"):
        preds = []
        start = time.time()

        for text in df[text_col].astype(str):
            ents = self._extract_entities(text)
            preds.append(ents)

        end = time.time()

        total_time = end - start
        latency = total_time / max(1, len(df))
        throughput = len(df) / max(1e-9, total_time)

        return preds, latency, throughput
    
    # 3. Entity Density
    def entity_density(self, preds, dataset):
        densities = []
        for ents, wc in zip(preds, dataset["word_count"]):
            densities.append(len(ents) / max(1, wc))
        return np.mean(densities)
            
    # 4. Entity Diversity
    def entity_diversity(self, preds):
        all_ents = [e["text"].lower() for doc in preds for e in doc]
        if len(all_ents) == 0:
            return 0.0
        return len(set(all_ents)) / len(all_ents)
    
    # 5. Label Entropy
    def label_entropy(self, preds):
        labels = [e["label"] for doc in preds for e in doc]
        if len(labels) == 0:
            return 0.0
        counts = np.array(list(Counter(labels).values()))
        probs = counts / counts.sum()
        return entropy(probs)
    
    # 6. Final score
    def final_score(self, m, weights=None):
        weights = {
            "density": 0.3,
            "diversity": 0.25,
            "entropy": 0.2,
            "latency" : -0.15,
            "throughput": 0.1,
        }
        
        score = 0
        for k,w in weights.items():
            score += m[k]*w
        
        return score
    
    # Main Evaluation
    def evaluate(self, df, text_col="Content"):
        preds, latency, throughput = self.run_model(df, text_col)

        metrics = {
            "density": self.entity_density(preds, df),
            "diversity": self.entity_diversity(preds),
            "entropy": self.label_entropy(preds),
            "latency": latency,
            "throughput": throughput,
        }

        metrics["score"] = self.final_score(metrics)
        
        return metrics
    

In [ ]:
nlp = spacy.load("en_core_web_sm")
sample = data.sample(10)

evaluator = NEREvaluator(model=nlp, model_type="spacy")
metrics = evaluator.evaluate(sample)

for k,v in metrics.items():
    print(f"{k:15s}: {round(v, 4)}")

##### Approx evaluation for spacy small model:

|   Metrics      |  Value  |
|--------------- |---------|
| density        | 0.1124  |
| diversity      | 0.3437  |
| entropy        | 2.1273  |
| latency        | 0.0554  |
| throughput     | 18.0394 |
| score          | 2.3407  |

We have already designed our semantic search and clustering pipeline but <br>
maybe we can improve them using NER in the future.

In [ ]:
eval = NEREvaluator(model=tagger, model_type="flair")
metrics = eval.evaluate(sample)

for k,v in metrics.items():
    print(f"{k:15s}: {round(v, 4)}")